In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("Python Spark SQL basic example")
    .config("spark.executor.memory", "512M")
    .config("spark.executor.cores", "1")
    .master("spark://master:7077")    #mai multe la https://spark.apache.org/docs/latest/configuration.html
    .getOrCreate()
)
spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/16 11:03:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
from pyspark.sql.functions import col, concat_ws

news_df = spark.read.csv(
    "/user/ubuntu/dataset/news.tsv",
    sep="\t",
    inferSchema=True,
    header=False
)

news_df = news_df.toDF(
    "news_id",
    "category",
    "subcategory",
    "title",
    "abstract",
    "url",
    "title_entities",
    "abstract_entities"
)

news_df.select("news_id", "title", "abstract").show(5, truncate=False)


+-------+----------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|news_id|title                                                                 |abstract                                                                                                                                                                                            |
+-------+----------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|N88753 |The Brands Queen Elizabeth, Prince Charles, and Prince Philip Swear By|Shop the notebooks, jackets, and more that the royals can't live without.             

In [8]:
from pyspark.sql.functions import col, concat_ws, lower, regexp_replace, udf
from pyspark.sql.types import ArrayType, StringType
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover
import nltk
from nltk.stem import WordNetLemmatizer
from pyspark.ml.feature import CountVectorizer

nltk.download('wordnet')

news_text = news_df.withColumn(
    "text",
    concat_ws(" ", col("title"), col("abstract"))
).select("news_id", "text")

cleaned = news_text.withColumn(
    "clean_text",
    regexp_replace(lower(col("text")), r"[^a-z\s]", " ")
)

tokenizer = RegexTokenizer(inputCol="clean_text", outputCol="tokens", pattern="\\s+", minTokenLength=2)
tokenized = tokenizer.transform(cleaned)

remover = StopWordsRemover(inputCol="tokens", outputCol="filtered_tokens")
filtered = remover.transform(tokenized)

lemmatizer = WordNetLemmatizer()

def lemmatize_tokens(tokens):
    return [lemmatizer.lemmatize(token) for token in tokens]

lemmatize_udf = udf(lemmatize_tokens, ArrayType(StringType()))
lemmatized = filtered.withColumn("lemmatized_tokens", lemmatize_udf(col("filtered_tokens")))

lemmatized.select("lemmatized_tokens").show(5, truncate=False)

cv = CountVectorizer(
    inputCol="lemmatized_tokens",
    outputCol="features",
    vocabSize=10000,
    minDF=5
)

cv_model = cv.fit(lemmatized)
vectorized = cv_model.transform(lemmatized)
vectorized.select("news_id", "features").show(5)


[nltk_data] Downloading package wordnet to /home/ubuntu/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/ubuntu/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
                                                                                

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|lemmatized_tokens                                                                                                                                                                                              |
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|[brand, queen, elizabeth, prince, charles, prince, philip, swear, shop, notebook, jacket, royal, live, without]                                                                                                |
|[walmart, slash, price, last, generation, ipads, apple, new, ipad, release, bring, big, deal, last, year, model]                                               

+-------+--------------------+
|news_id|            features|
+-------+--------------------+
| N88753|(10000,[263,267,9...|
| N45436|(10000,[0,1,26,79...|
| N23144|(10000,[25,139,94...|
| N86255|(10000,[17,25,57,...|
| N93187|(10000,[16,74,166...|
+-------+--------------------+
only showing top 5 rows



In [10]:
from pyspark.ml.clustering import LDA
from pyspark.sql.functions import udf
from pyspark.sql.types import ArrayType, DoubleType

lda = LDA(
    k=20,
    maxIter=20,
    featuresCol="features",
    seed=42
)

lda_model = lda.fit(vectorized)

topics = lda_model.describeTopics(7)

vocab = cv_model.vocabulary

def topic_words(termIndices):
    return [vocab[i] for i in termIndices]

topics_words = topics.rdd.map(
    lambda row: (row.topic, topic_words(row.termIndices))
).toDF(["topic", "words"])

topics_words.show(truncate=False)

article_topics = lda_model.transform(vectorized)

def round_probs(vec):
    return [round(float(x), 4) for x in vec]

round_udf = udf(round_probs, ArrayType(DoubleType()))

article_topics.select(
    "news_id",
    round_udf(col("topicDistribution"))
).show(5, truncate=False)




+-----+------------------------------------------------------------+
|topic|words                                                       |
+-----+------------------------------------------------------------+
|0    |[fire, firefighter, los, angeles, acre, county, bevin]      |
|1    |[fort, worth, bloomberg, new, pete, presidential, buttigieg]|
|2    |[top, photo, business, best, spot, plane, city]             |
|3    |[adoption, teacher, pet, miami, strike, chicago, marathon]  |
|4    |[week, weather, today, brown, v, steelers, game]            |
|5    |[christmas, holiday, train, vegan, subway, tree, fall]      |
|6    |[million, kansa, microsoft, pro, longhorn, raven, arkansas] |
|7    |[police, man, year, said, old, apartment, crash]            |
|8    |[new, state, trump, fire, news, say, house]                 |
|9    |[car, race, cup, nascar, driver, speedway, series]          |
|10   |[year, said, school, home, one, police, county]             |
|11   |[season, game, injury, team

26/01/16 11:23:10 WARN DAGScheduler: Broadcasting large task binary with size 1720.3 KiB
[Stage 146:>                                                        (0 + 1) / 1]

+-------+----------------------------------------------------------------------------------------------------------------------------------------------------------------+
|news_id|round_probs(topicDistribution)                                                                                                                                  |
+-------+----------------------------------------------------------------------------------------------------------------------------------------------------------------+
|N88753 |[0.0033, 0.0033, 0.0033, 0.0033, 0.0034, 0.0033, 0.0033, 0.0035, 0.0037, 0.0033, 0.0041, 0.0035, 0.0032, 0.5363, 0.0033, 0.0033, 0.4024, 0.0033, 0.0036, 0.0033]|
|N45436 |[0.0029, 0.0029, 0.0029, 0.0029, 0.003, 0.0029, 0.0029, 0.0031, 0.9433, 0.0029, 0.0036, 0.0031, 0.0028, 0.0033, 0.0029, 0.0029, 0.003, 0.0029, 0.0032, 0.0029]  |
|N23144 |[0.0033, 0.0033, 0.0033, 0.0033, 0.0034, 0.0033, 0.1276, 0.0035, 0.0037, 0.0033, 0.0041, 0.0035, 0.5748, 0.0038, 0.0033, 0.0033, 0.0034,

In [11]:

sample_articles = article_topics.join(
    news_df.select("news_id", "title", "abstract"),
    on="news_id",
    how="left"
).select("news_id", "title", "abstract", "topicDistribution") \
.sample(withReplacement=False, fraction=0.01, seed=42)
sample_articles = sample_articles.limit(5).collect()


topic_dict = {row['topic']: topic_words(row['termIndices']) for row in topics.collect()}

def format_top3(topic_dist, n=3):
    top_indices = sorted(range(len(topic_dist)), key=lambda i: topic_dist[i], reverse=True)[:n]
    result = []
    for i in top_indices:
        prob = topic_dist[i]
        words = ", ".join(topic_dict[i])
        result.append(f"Topic {i} -> {prob:.3f} - words: {words}")
    return "\n".join(result)

def format_article(title, abstract, topic_dist):
    top3 = format_top3(topic_dist)
    return f"{title}:\n{abstract}\n\nTop 3 topics:\n{top3}"

for row in sample_articles:
    print(f"news_id: {row.news_id}")
    print(format_article(row.title, row.abstract, row.topicDistribution))
    print("\n" + "-"*80 + "\n")


26/01/16 11:28:05 WARN DAGScheduler: Broadcasting large task binary with size 1718.8 KiB
26/01/16 11:28:22 WARN DAGScheduler: Broadcasting large task binary with size 1742.9 KiB
                                                                                

news_id: N101671
Goo Goo Dolls vocalist gets candid ahead of Des Moines show:
Grammy-nominated band Goo Goo Dolls is set to perform at Hoyt Sherman Place Sunday, Nov.3. Takac said that fans could expect new records and hits.

Top 3 topics:
Topic 13 -> 0.788 - words: new, state, year, time, one, first, michigan
Topic 10 -> 0.177 - words: year, said, school, home, one, police, county
Topic 8 -> 0.002 - words: new, state, trump, fire, news, say, house

--------------------------------------------------------------------------------

news_id: N102077
Kid Care Report: Day care off Chaffee Road cited for violations linked to sleep practices:
Every week Action News Jax investigates local child care centers to show you how they measure up to state guidelines. In this week's Kid Care report, Letisha Bereola shows you a day care with violations linked to safe sleep practices.

Top 3 topics:
Topic 8 -> 0.636 - words: new, state, trump, fire, news, say, house
Topic 10 -> 0.273 - words: year, said,